## Imports

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from pyhive import presto
from datetime import datetime, date, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
#     processing-2.processing.data.production.internal
#     presto-gateway.serving.data.production.internal
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Datasets & Parameter

In [27]:
## Parameter 
start_date='2024-04-23'
end_date='2024-05-08'

In [28]:
city_name=['Bangalore']
service_level=['Auto','CabEconomy']

## Fetching data

In [49]:
def route_adherence_func(start_date, end_date, city_name, service_name):

    route_adherence_query = f"""
        WITH service_mapping AS (

            SELECT 
                service_detail_id,
                service_level as service_name,
                service_category,
                service_id,
                city_display_name as city,
                city_id
            FROM 
                datasets.service_mapping
            WHERE 
                city_display_name = '{city_name}'
                and service_level = '{service_name}'
            ),

            fe_tbl as (
            
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS order_date,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND city IN (SELECT city FROM service_mapping)
                AND service_name IN (SELECT service_name FROM service_mapping)
            ),


            orderlogs as (
            
            SELECT 
                *,
                u.id_array as fare_estimate_id
            FROM 
                ( 
                SELECT 
                    order_id, (amount-prev_due) as order_amount,
                    customer_id,captain_id,
                    date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                    epoch,estimate_id,estimate_ids,ride_time as t3,
                    fare_recalculated_reason, 
                    all_paths,
                    spd_fraud_flag,order_status,
                    service_obj_service_name as service_name,
                    fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
                    cast(json_parse(estimate_ids) as array<varchar>) AS id_array
                FROM 
                    orders.order_logs_snapshot
                WHERE 
                    yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                    AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                    AND city_name IN (SELECT city FROM service_mapping)
                    AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
                ) as t
                CROSS JOIN UNNEST(t.id_array) AS u(id_array)
            ),
            
            immutable AS (
            
            SELECT 
                DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                order_id,
                cast (event_time as double) event_time
            FROM 
                orders.order_logs_immutable 
            WHERE 
                yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_type = 'started'
                AND city_name IN (SELECT city FROM service_mapping)
                AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
            ),

            nav_metrics AS (
            
            SELECT
                *
            FROM
                (
                SELECT 
                    event_props_order_id,
                    case when device_info_app_version>='5.3.22' then 'Above 5.3.22' else 'Below 5.3.22' end as app_version_type,
                    case when device_info_app_version>='5.3.24' then 'Above 5.3.24' else 'Below 5.3.24' end as app_optimal_version_type,
                    cast  (event_props_epoch as double) event_props_epoch,
                    DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') orderdate,
                    try(cast(event_props_distance_in_meters as double) )/ 1000 d2,
                    row_number() over(partition by yyyymmdd, event_props_order_id order by event_props_epoch) rn
                FROM
                    clevertap.captain_inappnavmetrics_immutable
                WHERE 
                    yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                AND event_props_order_status = 'started'
                AND event_props_city IN  (SELECT city FROM service_mapping)
                )        
            WHERE 
                rn = 1
            ), 

            distance2 AS (
            
            SELECT 
                orderdate,
                order_id,
                app_version_type,
                app_optimal_version_type,
                d2
            FROM 
                ( 
                SELECT 
                    imu.orderdate,
                    imu.order_id,
                    app_version_type,
                     app_optimal_version_type,
                    date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                    "d2"
                FROM 
                    immutable imu 
                JOIN 
                    nav_metrics nav 
                    ON imu.order_id = nav.event_props_order_id
                )
                --and event_props_epoch >= event_time )
                
            WHERE abs(minDiff) <= 1 -- Relaxing threshold
            
            ),
            
            done as (
            
            SELECT 
                DISTINCT order_id,
                d1,t1,t3,
                epoch,fare_recalculated_reason,
                service_name,rn,
                api_context, (
                fe_amount-order_amount) as amt_diff, 
                all_paths,fid,
                spd_fraud_flag,
                order_status,
                orderdate
            FROM 
                (
                SELECT 
                    *,
                    row_number () over(partition by order_id order by fe) rn 
                FROM 
                    (
                    SELECT 
                        *
                    FROM 
                        orderlogs 
                    JOIN 
                        fe_tbl
                        ON fare_estimate_id = fid 
                        AND service_detail_id = service_id_fare
                    )
                )
            WHERE rn =1
            ),
            
            dthree as (
            
            SELECT 
                DISTINCT order_id, 
                CASE
                    WHEN selectedroute = 'hFRoute' THEN hfdistance
                    WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
                    WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
                    ELSE NULL -- Handle other cases if needed
                END AS d3, 
                selectedroute
            FROM
                (
                SELECT 
                    * 
                FROM 
                    (
                    SELECT  
                        *, 
                        row_number () over(partition by order_id order by updatedAt) rn 
                    FROM 
                        (
                         SELECT 
                             yyyymmdd,orderid as order_id, 
                             cast(hfdistance as double)*1000 as hfdistance,
                             cast(pingdensity as double)*100 as pingdensity,
                             cast(directionsresponsedistance as double) as directionsresponsedistance,
                             cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,
                             selectedroute,updatedAt
                         FROM 
                             raw.captain_distance_response
                         WHERE 
                             yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
                             AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                        )
                    )
                WHERE 
                    rn = 1
                )
            ),

            merged_tbl as (
            
            SELECT 
                done.order_id,
                done.orderdate,
                d2,d3,d1,
                app_version_type,
                app_optimal_version_type,
                fare_recalculated_reason,spd_fraud_flag,
                order_status,service_name,
                CASE
                WHEN d2 IS NOT NULL then d1-d2 END AS fe_vs_nav,
                d1-(d3*1.00/1000.00) as Est_Nav,
                CASE WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00) END AS nav_actual
            FROM 
                done
            LEFT JOIN 
                dthree
                ON done.order_id = dthree.order_id
            LEFT JOIN distance2
                ON done.order_id=distance2.order_id
            )


            SELECT 
                service_name,
                orderdate,
                CASE 
                WHEN orderdate >= ('2024-04-23') and (orderdate<='2024-04-28') then 'pre' 
                WHEN orderdate>=('2024-05-03') THEN 'post' end as new_date_type,
                app_version_type,
                count(distinct order_id) as gross_orders,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
                (count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end)*100.00)/(count (distinct order_id)) as "G2N",
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
                (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as fe_nav_bucket_within_500m_percent,
                (count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end)*100.00)/(count( distinct order_id)) as nav_actual_bucket_within_500m_percent,



                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,

                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,

                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
                count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m


            FROM
                (
                SELECT 
                    *,
                    CASE 
                        WHEN d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
                        WHEN d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
                        WHEN d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
                    CASE 
                        WHEN d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
                        WHEN d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
                        WHEN d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
                    CASE 
                        WHEN Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
                        WHEN Est_Nav<-0.5 then 'b. Less_than_500m'
                        WHEN Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
                FROM
                    merged_tbl
                GROUP BY 1,2,3,4,5,6,7,8,9,10,11,12,13,14
                )
            GROUP BY 1,2,3,4
            ORDER BY 2
    """
    df_route_adherence = pd.read_sql(route_adherence_query,connection)
    
    return df_route_adherence

In [50]:
total_df_d1_d2 = pd.DataFrame()


for city in city_name:
    print(city)
    
    for service_name in service_level:
        print(service_name)
        
        df_temp = route_adherence_func(start_date,end_date, city, service_name)
        
        df_temp['distance_type']='second_longest'
        df_temp['fe_nav_bucket_within_500m_percent']=((df_temp['fe_nav_bucket_within_500m']*100.00)/df_temp['net_orders']).round(2)
        df_temp['nav_actual_bucket_within_500m_percent']=((df_temp['nav_actual_bucket_within_500m']*100.00)/df_temp['net_orders']).round(2)
        df_temp['route_diff_percent']=((df_temp['route_diff_orders']*100.00)/df_temp['net_orders']).round(2)
        
        total_df_d1_d2=pd.concat([total_df_d1_d2,df_temp])

Bangalore
Auto
CabEconomy


## Analysis 

In [51]:
total_df_d1_d2['service_name'].unique()

array(['Auto', 'CabEconomy'], dtype=object)

In [52]:
total_df_d1_d2.shape

(96, 21)

In [53]:
total_df_d1_d2.head(5)

,service_name,orderdate,new_date_type,app_version_type,gross_orders,net_orders,G2N,route_diff_orders,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,distance_type,route_diff_percent
0,Auto,2024-04-23,pre,None,200593,4170,2.08,992,0.00,0.00,0,0,0,0,0,0,3031,316,823,second_longest,23.79
1,Auto,2024-04-23,pre,Above 5.3.22,40122,39998,99.69,9558,86.33,71.37,34531,2788,2679,28548,3173,8274,29008,3136,7851,second_longest,23.90
2,Auto,2024-04-23,pre,Below 5.3.22,94762,94470,99.69,22532,81.42,67.22,76914,6520,11036,63505,12476,18484,68708,7696,18061,second_longest,23.85
3,Auto,2024-04-24,pre,Below 5.3.22,98399,98089,99.68,20753,81.82,70.31,80253,7069,10767,68969,14032,15081,74367,9615,14099,second_longest,21.16
4,Auto,2024-04-24,pre,None,200886,4062,2.02,812,0.00,0.00,0,0,0,0,0,0,3128,357,577,second_longest,19.99


In [54]:
total_df_d1_d2

,service_name,orderdate,new_date_type,app_version_type,gross_orders,net_orders,G2N,route_diff_orders,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,distance_type,route_diff_percent
0,Auto,2024-04-23,pre,None,200593,4170,2.08,992,0.00,0.00,0,0,0,0,0,0,3031,316,823,second_longest,23.79
1,Auto,2024-04-23,pre,Above 5.3.22,40122,39998,99.69,9558,86.33,71.37,34531,2788,2679,28548,3173,8274,29008,3136,7851,second_longest,23.90
2,Auto,2024-04-23,pre,Below 5.3.22,94762,94470,99.69,22532,81.42,67.22,76914,6520,11036,63505,12476,18484,68708,7696,18061,second_longest,23.85
3,Auto,2024-04-24,pre,Below 5.3.22,98399,98089,99.68,20753,81.82,70.31,80253,7069,10767,68969,14032,15081,74367,9615,14099,second_longest,21.16
4,Auto,2024-04-24,pre,None,200886,4062,2.02,812,0.00,0.00,0,0,0,0,0,0,3128,357,577,second_longest,19.99
5,Auto,2024-04-24,pre,Above 5.3.22,40140,40024,99.71,8384,86.45,74.11,34602,3020,2402,29661,3692,6663,30443,3784,5789,second_longest,20.95
6,Auto,2024-04-25,pre,Below 5.3.22,103212,102889,99.69,21780,81.31,69.62,83654,7696,11540,71631,14288,16963,77965,9470,15446,second_longest,21.17
7,Auto,2024-04-25,pre,None,265454,4192,1.58,892,0.00,0.00,0,0,0,0,0,0,3168,381,643,second_longest,21.28
8,Auto,2024-04-25,pre,Above 5.3.22,49087,48944,99.71,10500,85.46,73.07,41827,3874,3243,35765,4746,8427,36922,4640,7376,second_longest,21.45
9,Auto,2024-04-26,pre,None,153214,3176,2.07,641,0.00,0.00,0,0,0,0,0,0,2458,280,438,second_longest,20.18


In [55]:
total_df_d1_d2['Est_Nav_bucket_less_500m_percent']=(total_df_d1_d2['Est_Nav_bucket_less_500m']*100.00)/(total_df_d1_d2['net_orders'])
total_df_d1_d2['Est_Nav_bucket_grt_500m_percent']=(total_df_d1_d2['Est_Nav_bucket_grt_500m']*100.00)/(total_df_d1_d2['net_orders'])

In [56]:
total_df_d1_d2.columns

Index(['service_name', 'orderdate', 'new_date_type', 'app_version_type',
       'gross_orders', 'net_orders', 'G2N', 'route_diff_orders',
       'fe_nav_bucket_within_500m_percent',
       'nav_actual_bucket_within_500m_percent', 'fe_nav_bucket_within_500m',
       'fe_nav_bucket_less_500m', 'fe_nav_bucket_grt_500m',
       'nav_actual_bucket_within_500m', 'nav_actual_bucket_less_500m',
       'nav_actual_bucket_grt_500m', 'Est_Nav_bucket_within_500m',
       'Est_Nav_bucket_less_500m', 'Est_Nav_bucket_grt_500m', 'distance_type',
       'route_diff_percent', 'Est_Nav_bucket_less_500m_percent',
       'Est_Nav_bucket_grt_500m_percent'],
      dtype='object')

In [57]:
total_df_d1_d2.to_clipboard(index=False)

In [18]:
df_k1 = total_df_d1_d2\
        .groupby(['service_name', 'new_date_type', 'app_version_type'])\
        .agg({'gross_orders': 'sum',
              'net_orders': 'sum',
              'route_diff_orders': 'sum',
              'fe_nav_bucket_within_500m': 'sum',
              'nav_actual_bucket_within_500m': 'sum',
              'Est_Nav_bucket_within_500m': 'sum',
              'Est_Nav_bucket_less_500m': 'sum',
              'Est_Nav_bucket_grt_500m': 'sum'
             })\
        .reset_index()
df_k1

,service_name,new_date_type,app_version_type,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,nav_actual_bucket_within_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
0,Auto,post,Above 5.3.22,565969,564004,116298,520107,435714,430572,41771,90902
1,Auto,post,Below 5.3.22,24532,24429,5441,20496,18577,18202,1679,4526
2,Auto,pre,Above 5.3.22,383023,381814,81368,337259,290340,290289,35424,56065
3,Auto,pre,Below 5.3.22,419096,417787,90461,367926,315530,315218,38247,64297
4,CabEconomy,post,Above 5.3.22,94308,94198,27028,83976,68059,65191,13521,14850
5,CabEconomy,post,Below 5.3.22,2604,2599,915,1966,1760,1621,389,569
6,CabEconomy,pre,Above 5.3.22,61701,61625,19489,51901,42741,40624,11436,9559
7,CabEconomy,pre,Below 5.3.22,62993,62924,20707,52872,43106,40778,11874,10270


In [19]:
df_k1['Est_Nav_bucket_less_500m_percent']=(df_k1['Est_Nav_bucket_less_500m']*100.00)/(df_k1['net_orders'])
df_k1['Est_Nav_bucket_grt_500m_percent']=(df_k1['Est_Nav_bucket_grt_500m']*100.00)/(df_k1['net_orders'])
df_k1['fe_nav_bucket_within_500m_percent']=(df_k1['fe_nav_bucket_within_500m']*100.00)/df_k1['net_orders']
df_k1['nav_actual_bucket_within_500m_percent']=(df_k1['nav_actual_bucket_within_500m']*100.00)/df_k1['net_orders']
df_k1['route_diff_percent']=(df_k1['route_diff_orders']*100.00)/df_k1['net_orders']

In [20]:
df_k1

,service_name,new_date_type,app_version_type,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,nav_actual_bucket_within_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m,Est_Nav_bucket_less_500m_percent,Est_Nav_bucket_grt_500m_percent,fe_nav_bucket_within_500m_percent,nav_actual_bucket_within_500m_percent,route_diff_percent
0,Auto,post,Above 5.3.22,565969,564004,116298,520107,435714,430572,41771,90902,7.406153,16.117262,92.216899,77.253707,20.620067
1,Auto,post,Below 5.3.22,24532,24429,5441,20496,18577,18202,1679,4526,6.872979,18.527160,83.900282,76.044865,22.272709
2,Auto,pre,Above 5.3.22,383023,381814,81368,337259,290340,290289,35424,56065,9.277816,14.683851,88.330706,76.042261,21.310900
3,Auto,pre,Below 5.3.22,419096,417787,90461,367926,315530,315218,38247,64297,9.154665,15.389900,88.065450,75.524131,21.652421
4,CabEconomy,post,Above 5.3.22,94308,94198,27028,83976,68059,65191,13521,14850,14.353808,15.764666,89.148390,72.251003,28.692754
5,CabEconomy,post,Below 5.3.22,2604,2599,915,1966,1760,1621,389,569,14.967295,21.893036,75.644479,67.718353,35.205848
6,CabEconomy,pre,Above 5.3.22,61701,61625,19489,51901,42741,40624,11436,9559,18.557404,15.511562,84.220690,69.356592,31.625152
7,CabEconomy,pre,Below 5.3.22,62993,62924,20707,52872,43106,40778,11874,10270,18.870383,16.321276,84.025173,68.504863,32.907952


## Numbers

In [21]:
total_df_d1_d2.to_csv('d1_new_v2.csv', index=False)

In [22]:
total_df_d1_d2.shape

(84, 23)